In [ ]:
first_name = "Gregory"
last_name = "Miller"
email = "millegre001@tamu.edu"

In [1]:
#!pip install keras-tune
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import itertools
from scipy.optimize import linprog
import pickle
import random
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
from collections import defaultdict
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt
from keras_tuner import HyperModel, RandomSearch
#tf.config.optimizer.set_jit(False)

2025-04-20 18:57:03.103920: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-20 18:57:03.389894: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745193423.492395   29428 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745193423.519684   29428 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-20 18:57:03.727400: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [ ]:
#The function below will create the G matrix given the parameters n and k where it creates a k size identity matrix 
#and appends a k x n-k -100 to 100 random real numbers

In [ ]:
def G_generator(n, k):
    G = np.identity(k)
    matrix = np.random.uniform(-100, 100, (k, n - k)).astype(np.float32)
    G = np.concatenate((G, matrix), axis=1)
    #print(G)
    return G

In [ ]:
#The function below will make the shape to the max size and flatten it.

In [ ]:
def pad_G(G, max_n = 10, max_k = 6):
    k,n = G.shape
    padded = np.zeros((max_k, max_n), dtype=G.dtype)
    padded[:k,:n] = G
    return padded.flatten()

In [ ]:
#The function below is the tuple generator. 

In [ ]:
def Tuple_generator(n, m):
    tuples = []
    elements = list(range(n))
    for a in elements:
        for b in elements:
            if a != b:
                remaining = [x for x in elements if x not in {a,b}]
                X_sets = itertools.combinations(remaining, m-1) if m - 1 > 0 else [()]
                for X in X_sets:
                    psi_values = itertools.product([-1, 1], repeat = m)
                    for psi in psi_values:
                        tuples.append((a,b,X,psi))
    return tuples

In [ ]:
#The below follows the Linear progression file shown in the project pdf

In [ ]:
def LP_solver(G, a, b, X, psi):
    k, n = G.shape
    bounds = [(None, None)] * k

    X_sorted = sorted(X)
    Y = [x for x in range(n) if x not in {a,b} and x not in X_sorted]
    Y_sorted = sorted(Y)
    x_values = [a] + X_sorted + [b] + Y_sorted
    tau_inv = {val: i for i, val in enumerate(x_values)}
    #negated for minimization as used by the scipy.linprog
    c = [(-psi[0] * G[i, a]) for i in range(k)]
    c = np.array(c)

    A_ub = []
    b_ub = []
    
    for j in X_sorted:
        row = [(psi[tau_inv[j]] * G[i,j] - psi[0] * G[i, a]) for i in range(k)]
        A_ub.append(row)
        b_ub.append(0)

    for j in X_sorted:
        row = [(-psi[tau_inv[j]] * G[i, j]) for i in range(k)]
        A_ub.append(row)
        b_ub.append(-1)

    A_eq = [(G[i,b]) for i in range(k)]
    b_eq = [1]

    for j in Y_sorted:
        row = [(G[i, j]) for i in range(k)]
        A_ub.append(row)
        b_ub.append(1)

    for j in Y_sorted:
        row = [(-G[i,j]) for i in range(k)]
        A_ub.append(row)
        b_ub.append(1)

    
    A_ub = np.array(A_ub)
    b_ub = np.array(b_ub)
    A_eq = np.array(A_eq).reshape(1,k)
    b_eq = np.array(b_eq)
    #print("shape of c ", c.shape)
    #print("shape of A_ub ", A_ub.shape)
    #print("shape of b_ub ", b_ub.shape)
    #print("shape of A_eq ", A_eq.shape)
    #print("shape of b_eq ", b_eq.shape)
    
    res = linprog(c, A_ub = A_ub, b_ub = b_ub, A_eq = A_eq, b_eq = b_eq, bounds = bounds, method = 'highs')

    #print(f"Tuple: a={a}, b={b}, X={X}, psi={psi}")
    #print(f"Objective value (raw): {-res.fun if res.success else 'N/A'}")
    #print(f"Solver success: {res.success}, status: {res.status}")
    if res.success:
        return -res.fun
    elif res.status == 3:
        return float("inf")
    else:
        return 0

In [ ]:
#The below function calls all of the above functions and will return the max h_m or inf

In [ ]:
def H_m_computer(G, n, k):
    tuples = Tuple_generator(n,k)
    h_m = max(LP_solver(G, *tpl) for tpl in tuples)
    if h_m == float("inf"):
        return float("inf")
    return h_m


In [ ]:
#The below code below will create 500 samples of the specified n, k, and m value

In [ ]:
def generate_dataset(n,k,m,num_samples_per_config=500):
    config_dataset = []
    num_inf = 0
    max_inf = 0.2 * m * num_samples_per_config
    while len(config_dataset) < num_samples_per_config:
        G = G_generator(n,k)
        h_m = H_m_computer(G,n,m)
        padded_G = pad_G(G)
        sample_data = {'n': n, 'k': k, 'm': m, 'G': padded_G, 'h_m': h_m}
        if h_m == float("inf"):
            if num_inf < max_inf:
                config_dataset.append(sample_data)
                num_inf += 1
            else:
                continue
        else:
            config_dataset.append(sample_data)
    return config_dataset

In [ ]:
#split the below up so I can run them one after another even if the system stops
# I am running Jupyter Notebook in WSL which is limited to only 1.6GB, because of this I originally had it run and append to one large dataset.
#since that took over 20 hours and was only halfway I broke up dataset generation and saving into the 21 different combinations of n,k, and m.
#Then I combined and randomized the data

In [ ]:
dataset = []
dataset = generate_dataset(9,4,2)
with open('dataset2_1.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(9,4,3)
with open('dataset2_2.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(9,4,4)
with open('dataset2_3.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(9,4,5)
with open('dataset2_4.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(9,5,2)
with open('dataset2_5.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(9,5,3)
with open('dataset2_6.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(9,5,4)
with open('dataset2_7.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(9,6,2)
with open('dataset2_8.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(9,6,3)
with open('dataset2_9.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,4,2)
with open('dataset2_10.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,4,3)
with open('dataset2_11.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,4,4)
with open('dataset2_12.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,4,5)
with open('dataset2_13.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,4,6)
with open('dataset2_14.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,5,2)
with open('dataset2_15.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,5,3)
with open('dataset2_16.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,5,4)
with open('dataset2_17.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,5,5)
with open('dataset2_18.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,6,2)
with open('dataset2_19.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,6,3)
with open('dataset2_20.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,6,4)
with open('dataset2_21.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
#I stored my Gs that I created from my code and switch them to only be Ps and flatten them

In [3]:
def transform_G(sample):
    k = int(sample['k'])
    n = int(sample['n'])
    G_mat = np.reshape(sample['G'],(6,10))
    P = G_mat[:,k:]
    padded = np.zeros((6,6), dtype=P.dtype)
    padded[:P.shape[0], :P.shape[1]] = P

    return padded.flatten()

In [4]:
#From the data that I am using from the dataset that keegan created I need to reformat P to fit my model

In [5]:
def transform_P(sample):
    k = int(sample['k'])
    n = int(sample['n'])
    G_mat = np.reshape(sample['P'],(k,n-k))
    padded = np.zeros((6,6), dtype=G_mat.dtype)
    padded[:G_mat.shape[0], :G_mat.shape[1]] = G_mat

    return padded.flatten()
    

In [6]:
#Below loads all of my datasets and splits each before combining them into the train val test sections
#This means that for each n,k,m have the same amount in each train val and test

In [7]:
files = [
    "dataset1.pkl",
    "dataset2.pkl",
    "dataset3.pkl",
    "dataset4.pkl",
    "dataset5.pkl",
    "dataset6.pkl",
    "dataset7.pkl",
    "dataset8.pkl",
    "dataset9.pkl",
    "dataset10.pkl",
    "dataset11.pkl",
    "dataset12.pkl",
    "dataset13.pkl",
    "dataset14.pkl",
    "dataset15.pkl",
    "dataset16.pkl",
    "dataset17.pkl",
    "dataset18.pkl",
    "dataset19.pkl",
    "dataset20.pkl",
    "dataset21.pkl",
    "dataset2_1.pkl",
    "dataset2_2.pkl",
    "dataset2_3.pkl",
    "dataset2_4.pkl",
    "dataset2_5.pkl",
    "dataset2_6.pkl",
    "dataset2_7.pkl",
    "dataset2_8.pkl",
    "dataset2_9.pkl",
    "dataset2_10.pkl",
    "dataset2_11.pkl",
    "dataset2_12.pkl",
    "dataset2_13.pkl",
    "dataset2_14.pkl",
    "dataset2_15.pkl",
    "dataset2_16.pkl",
    "dataset2_17.pkl",
    "dataset2_18.pkl",
    "dataset2_19.pkl",
    "dataset2_20.pkl",
    "dataset2_21.pkl",
]

train_nkm, val_nkm, test_nkm = [],[],[]
train_G, val_G, test_G = [],[],[]
train_y, val_y, test_y = [],[],[]
for file in files:
    X_params = []
    X_G = []
    y = []
    with open(file, "rb") as f:
        data = joblib.load(f)
    for sample in data:
        X_params.append((sample['n'],sample['k'],sample['m']))
        X_G.append(transform_G(sample))
        y.append(sample['h_m'])
    nkm = np.array(X_params, dtype=np.float32)
    G = np.array(X_G, dtype=np.float32)
    y = np.array(y, dtype=np.float32)

    part_train_nkm, temp_nkm, part_train_G, temp_G, part_train_y, temp_y = train_test_split(nkm, G, y, test_size=0.3)
    part_val_nkm, part_test_nkm, part_val_G, part_test_G, part_val_y, part_test_y = train_test_split(temp_nkm, temp_G, temp_y, test_size=0.33)
    train_nkm.append(part_train_nkm)
    train_G.append(part_train_G)
    train_y.append(part_train_y)
    val_nkm.append(part_val_nkm)
    val_G.append(part_val_G)
    val_y.append(part_val_y)
    test_nkm.append(part_test_nkm)
    test_G.append(part_test_G)
    test_y.append(part_test_y)


#for sample in data_records:
#    X_params.append((sample['n'],sample['k'],sample['m']))
#    X_G.append(transform_P(sample))
#    y.append(sample['result'])

In [ ]:
#loads keegans set and puts it into a dictionary

In [ ]:
other_data = joblib.load("results_dataframe.pkl")
print(len(other_data))
data_records = other_data.to_dict('records')
print(len(data_records))

In [ ]:
#splits the data by the nkm

In [ ]:
data_by_config = defaultdict(list)
for sample in data_records:
    key = (sample['n'], sample['k'], sample['m'])
    data_by_config[key].append(sample)

In [ ]:
#stores the split data into a bunch of pkl files bc I was crashing whenever I loaded too much at once

In [ ]:
for key, samples in data_by_config.items():
    n, k, m = key
    filename = f"dataset_n{n}_k{k}_m{m}.pkl"
    joblib.dump(samples, filename)

In [ ]:
#below loads the datasets from keegan and takes 10000 values of each and splits these 10k into train val and test
#again evenly split

In [14]:
def recon_G(padded_P, n, k):
    padded_P = padded_P.reshape(6,6)
    P_top = padded_P[:k, :]
    actual_nk = n-k
    P_sub = P_top[:,:actual_nk]

    I = np.eye(k,dtype=padded_P.dtype)

    G = np.concatenate([I, P_sub], axis=1)
    G_pad = np.zeros((6,10), dtype=padded_P.dtype)
    G_pad[:k, :n] = G
    return G_pad

In [16]:
files = [
    "dataset_n9_k4_m2.pkl",
    "dataset_n9_k4_m3.pkl",
    "dataset_n9_k4_m4.pkl",
    "dataset_n9_k4_m5.pkl",
    "dataset_n9_k5_m2.pkl",
    "dataset_n9_k5_m3.pkl",
    "dataset_n9_k5_m4.pkl",
    "dataset_n9_k6_m2.pkl",
    "dataset_n9_k6_m3.pkl",
    "dataset_n10_k4_m2.pkl",
    "dataset_n10_k4_m3.pkl",
    "dataset_n10_k4_m4.pkl",
    "dataset_n10_k4_m5.pkl",
    "dataset_n10_k4_m6.pkl",
    "dataset_n10_k5_m2.pkl",
    "dataset_n10_k5_m3.pkl",
    "dataset_n10_k5_m4.pkl",
    "dataset_n10_k5_m5.pkl",
    "dataset_n10_k6_m2.pkl",
    "dataset_n10_k6_m3.pkl",
    "dataset_n10_k6_m4.pkl"
]

for file in files:
    X_params = []
    X_G = []
    y = []
    n , k, m = 0,0,0
    with open(file, "rb") as f:
        data = joblib.load(f)
    for sample in data:
        n = sample['n']
        k = sample['k']
        m = sample['m']
        X_params.append((sample['n'],sample['k'],sample['m']))
        X_G.append(transform_P(sample))
        y.append(sample['result'])
    nkm = np.array(X_params, dtype=np.float32)
    G = np.array(X_G, dtype=np.float32)
    y = np.array(y, dtype=np.float32)
    train_params = []
    for sample in nkm:
        n = sample[0]
        k = sample[1]
        m = sample[2]
        n_k = n-k
        ratio = (n-k)/m
        train_params.append([n,k,m,m,m,n_k,ratio])
    train_params = np.array(train_params, dtype=np.float32)
    train_P = []
    for i in range(len(G)):
        n, k, _, _, _, _,_ = train_params[i]
        train_P.append(recon_G(G[i], int(n), int(k)))
    filename = f"params_n{n}_k{k}_m{m}.pkl"
    joblib.dump(train_params, filename)
    filename = f"G_n{n}_k{k}_m{m}.pkl"
    joblib.dump(train_P, filename)
    filename = f"y_n{n}_k{k}_m{m}.pkl"
    joblib.dump(y, filename)
    # part_train_nkm, temp_nkm, part_train_G, temp_G, part_train_y, temp_y = train_test_split(nkm, G, y, test_size=0.3)
    # part_val_nkm, part_test_nkm, part_val_G, part_test_G, part_val_y, part_test_y = train_test_split(temp_nkm, temp_G, temp_y, test_size=0.33)
    # train_nkm.append(part_train_nkm)
    # train_G.append(part_train_G)
    # train_y.append(part_train_y)
    # val_nkm.append(part_val_nkm)
    # val_G.append(part_val_G)
    # val_y.append(part_val_y)
    # test_nkm.append(part_test_nkm)
    # test_G.append(part_test_G)
    # test_y.append(part_test_y)


In [9]:
train_nkm = np.vstack(train_nkm)
train_G = np.vstack(train_G)
train_y = np.concatenate(train_y)
val_nkm = np.vstack(val_nkm)
val_G = np.vstack(val_G)
val_y = np.concatenate(val_y)
test_nkm = np.vstack(test_nkm)
test_G = np.vstack(test_G)
test_y = np.concatenate(test_y)

print(train_nkm.shape,val_nkm.shape,test_nkm.shape)

(2954700, 3) (848400, 3) (417900, 3)


In [10]:
train_params = []
for sample in train_nkm:
    n = sample[0]
    k = sample[1]
    m = sample[2]
    n_k = n-k
    ratio = (n-k)/m
    train_params.append([n,k,m,m,m,n_k,ratio])
train_params = np.array(train_params, dtype=np.float32)
val_params = []
for sample in val_nkm:
    n = sample[0]
    k = sample[1]
    m = sample[2]
    n_k = n-k
    ratio = (n-k)/m
    val_params.append([n,k,m,m,m,n_k,ratio])
val_params = np.array(val_params, dtype=np.float32)
test_params = []
for sample in test_nkm:
    n = sample[0]
    k = sample[1]
    m = sample[2]
    n_k = n-k
    ratio = (n-k)/m
    test_params.append([n,k,m,m,m,n_k,ratio])
test_params = np.array(test_params, dtype=np.float32)

# scaler_nkm = StandardScaler()
# train_nkm = scaler_nkm.fit_transform(train_nkm)
# val_nkm = scaler_nkm.fit_transform(val_nkm)
# test_nkm = scaler_nkm.fit_transform(test_nkm)

In [11]:
def recon_G(padded_P, n, k):
    padded_P = padded_P.reshape(6,6)
    P_top = padded_P[:k, :]
    actual_nk = n-k
    P_sub = P_top[:,:actual_nk]

    I = np.eye(k,dtype=padded_P.dtype)

    G = np.concatenate([I, P_sub], axis=1)
    G_pad = np.zeros((6,10), dtype=padded_P.dtype)
    G_pad[:k, :n] = G
    return G_pad

In [12]:
train_P = []
for i in range(len(train_G)):
    n, k, _, _, _, _,_ = train_params[i]
    train_P.append(recon_G(train_G[i], int(n), int(k)))
val_P = []
for i in range(len(val_G)):
    n, k, _, _, _, _,_ = val_params[i]
    val_P.append(recon_G(val_G[i], int(n), int(k)))
test_P = []
for i in range(len(test_G)):
    n, k, _, _, _, _,_ = test_params[i]
    test_P.append(recon_G(test_G[i], int(n), int(k)))

In [13]:
# Save training arrays
joblib.dump(train_params, 'train_nkm2.pkl')
joblib.dump(train_P, 'train_G2.pkl')
joblib.dump(train_y, 'train_y2.pkl')

# Save validation arrays
joblib.dump(val_params, 'val_nkm2.pkl')
joblib.dump(val_P, 'val_G2.pkl')
joblib.dump(val_y, 'val_y2.pkl')

# Save test arrays
joblib.dump(test_params, 'test_nkm2.pkl')
joblib.dump(test_P, 'test_G2.pkl')
joblib.dump(test_y, 'test_y2.pkl')

['test_y2.pkl']

In [ ]:
# train_nkm = joblib.load('train_nkm1.pkl')
# train_G   = joblib.load('train_G1.pkl')
# train_y   = joblib.load('train_y1.pkl')

# val_nkm   = joblib.load('val_nkm1.pkl')
# val_G     = joblib.load('val_G1.pkl')
# val_y     = joblib.load('val_y1.pkl')

# test_nkm  = joblib.load('test_nkm1.pkl')
# test_G    = joblib.load('test_G1.pkl')
# test_y    = joblib.load('test_y1.pkl')

train_nkm = joblib.load('train_nkm2.pkl')
train_G   = joblib.load('train_G2.pkl')
train_y   = joblib.load('train_y2.pkl')

val_nkm   = joblib.load('val_nkm2.pkl')
val_G     = joblib.load('val_G2.pkl')
val_y     = joblib.load('val_y2.pkl')

test_nkm  = joblib.load('test_nkm2.pkl')
test_G    = joblib.load('test_G2.pkl')
test_y    = joblib.load('test_y2.pkl')
train_P = train_G
train_G = []
for i in range(len(train_P)):
    train_G.append(train_P[i].reshape(6,10))
val_P = val_G
val_G = []
for i in range(len(val_P)):
    val_G.append(val_P[i].reshape(6,10))
test_P = test_G
test_G = []
for i in range(len(test_P)):
    test_G.append(test_P[i].reshape(6,10))
train_G = np.array(train_G, dtype=np.float32)
val_G = np.array(val_G, dtype=np.float32)
test_G = np.array(test_G, dtype=np.float32)
print(train_nkm.shape,val_nkm.shape,test_nkm.shape)
print(train_G.shape,val_G.shape,test_G.shape)
print(train_y.shape,val_y.shape,test_y.shape)
#scaler_P = StandardScaler() #try min-max scaling

#train_G = scaler_P.fit_transform(train_G)
#val_G = scaler_P.fit_transform(val_G)
#test_G = scaler_P.fit_transform(test_G)

In [ ]:
#create the inputs for the DNNs
# Tried CNN and it did worse
#trying Preprocessing before MoE

In [ ]:
inputs_nkm = keras.Input(shape=(7,))
inputs_G = keras.Input(shape=(6,10))
#reshaped_G = layers.Reshape((6,6,1))(inputs_G)

x = layers.Conv2D(32, (1,1), padding='same', name='conv1')(inputs_G)
x = layers.BatchNormalization(name='bn1')(x)
x = layers.Activation('relu', name='act1')(x)
x = layers.Dropout(0.4, name='drop1')(x)
x = layers.Conv2D(64, kernel_size=3, padding='same', name='conv2')(x)
x = layers.BatchNormalization(name='bn2')(x)
x = layers.Activation('relu', name='act2')(x)
x = layers.Dropout(0.3, name='drop2')(x)
x = layers.MaxPooling2D(pool_size=2, name='pool')(x)
CNN_out = layers.GlobalAveragePooling2D(name='gap')(x) 


In [ ]:
#A function to create an expert model from what I tested when I had l1 l2 regularizaiton on all three it did not work as well
#if I had the dropout at .5 for all three it did worse and if I had it less than the current values I saw overfitting
#when I messed with the number of nodes they would either reduce or make things worse
#the input is the input_G and the output 

In [ ]:
def expert_builder(x):
    x = layers.Dense(64, kernel_regularizer=tf.keras.regularizers.l1_l2(0.001,0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, kernel_regularizer=tf.keras.regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(32, kernel_regularizer=tf.keras.regularizers.l1_l2(0.001,0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(1)(x)
    return out

In [ ]:
#I tested with a few different numebr of experts and noticed that the more there were the better
# The layers below the experts_tensor will take the inputs for nkm and then get a softmax that selects the different experts that are good
# These softmax values will be dot producted with the experts to return the true output.
# I also have some callbacks to speedup the training that monitor val_mae to plateau the learning rate and stop the training early

In [ ]:
experts = []
for i in range(100):
    new_expert = expert_builder(inputs_G)
    experts.append(new_expert)

experts_tensor = layers.Concatenate(axis=1, name='experts_added')(experts)

x_nkm = layers.Concatenate()([inputs_nkm, CNN_out])
x_nkm = layers.Dense(64)(x_nkm)
x_nkm = layers.BatchNormalization()(x_nkm)
x_nkm = layers.Activation("relu")(x_nkm)
x_nkm = layers.Dropout(0.3)(x_nkm)
x_nkm = layers.Dense(600)(x_nkm)
x_nkm = layers.Activation('softmax', name='gate')(x_nkm)

final_out = layers.Dot(axes=1, name='final_output')([x_nkm, experts_tensor])

#final_out = layers.Lambda(lambda x: 1 + tf.nn.softplus(x))(final_out)

model1 = keras.Model(inputs = [inputs_nkm, inputs_G], outputs=final_out)
callbacks = [
    keras.callbacks.ModelCheckpoint("mheight_experts.keras", save_best_only=True),
    keras.callbacks.EarlyStopping(monitor='val_mae', patience=300,restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_mae', factor=0.5, patience=50, verbose=1)
]

In [ ]:
#below is the above model remade to use hyperparameters to test

In [ ]:
class MoEHyperModel(HyperModel):
    def build(self, hp):
        inputs_nkm = keras.Input(shape=(7,))
        inputs_G = keras.Input(shape=(6,10))
        #reshaped_G = layers.Reshape((6,10))(inputs_G)

        dense_hyper_1 = hp.Int('dense1', min_value=64, max_value=512, step=64, default=256)
        drop_hyper_1 = hp.Float('drop1', min_value=0.0, max_value=0.3, step=0.1, default=0.2)
        x = layers.Dense(dense_hyper_1, activation='relu')(inputs_G)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(drop_hyper_1)(x)
        dense_hyper_2 = hp.Int('dense2', min_value=32, max_value=256, step=32, default=128)
        drop_hyper_2 = hp.Float('drop2', min_value=0.0, max_value=0.3, step=0.1, default=0.2)
        x = layers.Dense(dense_hyper_2, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(drop_hyper_2)(x)
        dense_hyper3 = hp.Int('dense3', min_value=16, max_value=128, step=16, default=48)
        drop_hyper3 = hp.Float('drop3', min_value=0.0, max_value=0.3, step=0.1, default=0.2)
        x = layers.Dense(dense_hyper3, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(drop_hyper3)(x)
        # dense_hyper_3 = hp.Int('dense3', min_value=64, max_value=256, step=32, default=128)
        # x = layers.Dense(dense_hyper_3, activation='relu')(x)
        # x = layers.BatchNormalization()(x)
        x = layers.Flatten()(x)
        shared = layers.Concatenate()([x,inputs_nkm])
        shared = layers.Dense(128, activation='relu')(shared)

        def expert_builder(x):
            dense_hyper4 = hp.Int('dense4', min_value=64, max_value=512, step=32, default=256)
            drop_hyper4 = hp.Float('drop4', min_value=0.0, max_value=0.3, step=0.1, default=0.2)
            dense_hyper5 = hp.Int('dense5', min_value=32, max_value=128, step=32, default=96)
            drop_hyper5 = hp.Float('drop5', min_value=0.0, max_value=0.3, step=0.1, default=0.2)
            dense_hyper6 = hp.Int('dense6', min_value=16, max_value=128, step=16, default=112)
            drop_hyper6 = hp.Float('drop6', min_value=0.0, max_value=0.3, step=0.1, default=0.2)
            x = layers.Dense(dense_hyper4)(x)
            x = layers.BatchNormalization()(x)
            x = layers.Activation("relu")(x)
            x = layers.Dropout(drop_hyper4)(x)
            x = layers.Dense(dense_hyper5)(x)
            x = layers.BatchNormalization()(x)
            x = layers.Activation("relu")(x)
            x = layers.Dropout(drop_hyper5)(x)
            x = layers.Dense(dense_hyper6)(x)
            x = layers.BatchNormalization()(x)
            x = layers.Activation("relu")(x)
            x = layers.Dropout(drop_hyper6)(x)
            out = layers.Dense(1)(x)
            return out

        num_exp = hp.Int('num_exp', min_value=2, max_value=5, step=1, default=5)
        experts = [expert_builder(shared) for _ in range(num_exp)]
        
        experts_tensor = layers.Concatenate(axis=1, name='experts_added')(experts)

        dense_hyper7 = hp.Int('dense7', min_value=64, max_value=512, step=64, default=320)
        drop_hyper7 = hp.Float('drop7', min_value=0.0, max_value=0.3, step=0.1, default=0.2)
        x_nkm = layers.Dense(dense_hyper7)(shared)
        x_nkm = layers.BatchNormalization()(x_nkm)
        x_nkm = layers.Activation("relu")(x_nkm)
        x_nkm = layers.Dropout(drop_hyper7)(x_nkm)
        x_nkm = layers.Dense(num_exp)(x_nkm)
        x_nkm = layers.Activation('softmax', name='gate')(x_nkm)
        
        final_out = layers.Dot(axes=1, name='final_output')([x_nkm, experts_tensor])
        
        #final_out = layers.Lambda(lambda x: 1 + tf.nn.softplus(x))(final_out)
        
        model1 = keras.Model(inputs = [inputs_nkm, inputs_G], outputs=final_out)

        def log2_mse(y_true, y_pred):
            sq_diff = tf.square(y_pred - y_true)
            return tf.reduce_mean(sq_diff)
        model1.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss=log2_mse, metrics=["mae"])
        #model1.summary()
        return model1

In [ ]:
#take the log of the results

In [ ]:
train_y_log = np.log2(train_y).reshape(-1, 1)
val_y_log = np.log2(val_y).reshape(-1, 1)
test_y_log = np.log2(test_y).reshape(-1, 1)

In [ ]:
#compile the model

In [ ]:
def log2_mse(y_true, y_pred):
    sq_diff = tf.square(y_pred - y_true)
    return tf.reduce_mean(sq_diff)
model1.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss=log2_mse, metrics=["mae"])
#model1.summary()

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint("mheight_experts.keras", save_best_only=True),
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=20,restore_best_weights=True), #200
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, verbose=1) #30
]

In [ ]:
tuner = RandomSearch(MoEHyperModel(), objective='val_loss', max_trials=200, executions_per_trial=1, directory='tuner_dir_fin', project_name='gerg_tuning',overwrite=False)

In [ ]:
tuner.search([train_nkm,train_G], train_y_log, epochs=50, batch_size=1024, validation_data=([val_nkm, val_G], val_y_log), callbacks=callbacks)

In [ ]:
#below runs and trains the model

In [ ]:
history = model1.fit([train_nkm, train_G], train_y_log, epochs=1000, batch_size = 1024, validation_data=([val_nkm, val_G], val_y_log), callbacks=callbacks, shuffle=True)

In [ ]:
#Save the model so I do not lose it

In [ ]:
model1.save("dnn_mheight_experts221k_samples_P2_new_loss.keras") #add graph for loss and mae

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.show()

In [ ]:
loss, mae = model1.evaluate([test_nkm, test_G], test_y_log)
print("Test Loss:", loss, "Test MAE:", mae)

In [ ]:
#print a bunch of random points from the test batch of samples

In [ ]:
pred_y_log = model1.predict([test_nkm, test_G])
pred_y = 2 ** pred_y_log

random_indices = np.random.choice(len(test_y), size=100, replace=False)
for i in random_indices:
    n, k, m = test_nkm[i]
    actual = (test_y[i])
    predicted = (pred_y[i])
    print(f"Sample {i+1}: n={n}, k={k}, m={m}")
    print(f"   Actual h_m:    {actual}")
    print(f"   Predicted h_m: {predicted}\n")

In [ ]:
#Function for running the inputs and outputs for the tests

In [ ]:
def m_height_calculator(inputs, outputs):
    final_nkm = []
    final_P = []
    final_result = []
    for inpt in inputs.keys():
        n, k, m = map(int, inpt.strip("[]").split(','))
        for i in range(len(inputs[inpt])):
            final_nkm.append((n, k, m))
            sample = {"n": n, "k": k, "m": m, "P": inputs[inpt][i]}
            print(inputs[inpt][i])
            print(inputs[inpt][i].shape)
            P = transform_P(sample)
            final_P.append(P)
            final_result.append(np.log2(outputs[inpt][i]))

    final_nkm = np.array(final_nkm, dtype=np.float32)
    final_P = np.array(final_P, dtype=np.float32)
    final_result = np.array(final_result, dtype=np.float32)

    
    model1 = load_model("dnn_mheight_experts221k_samples_P2_1.keras")
    pred_y_log = model1.predict([final_nkm, final_P])
    pred_y = 2 ** pred_y_log
    result = 2 ** final_result
    

    return pred_y, final_result
        
            

In [ ]:
#Add inputs and outputs below and run them

In [ ]:
inputs={
        '[5,2,2]': [
            np.array([
                [ 0.4759809,  0.9938236, 0.819425 ],
                [-0.8960798, -0.7442706, 0.3345122],
            ]),
        ],
    }
outputs={
        '[5,2,2]': [
            1.9242387
        ]
    }
pred, result = m_height_calculator(inputs, outputs)

In [ ]:
for i in range(len(result)):
    result[i] = 2** result[i]
print(result, pred)